# Benchmarking Datasets Load

## Benchmarks Data Source

Two Corpi
- Simple Small Manual List
- Hugging Face Dataset: 

## Benchmarks Created

Each corpus to a standard: 
- `source`, `split`, `id`, `text`
- One float column per axis of the representation

In [1]:
# Get the root path and data paths
#

from pathlib import Path

import pandas as pd


def repo_root(marker: str = "uv.lock") -> Path:
    """Nearest ancestor of the working directory containing *marker*."""
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / marker).is_file():
            return candidate
    raise FileNotFoundError(f"No {marker} found above {start}")

DATA_IN = repo_root() / "data_in"
HF_REPO = "brighter-dataset/BRIGHTER-emotion-categories"

SIMPLE_CSV = DATA_IN / "simple_text_ekman6_raw.csv"
BRIGHTER_DF = DATA_IN / "brighter_emotions_raw.parquet"

BENCH_SIMPLE = DATA_IN / "bench_simple_ekman6.parquet"
BENCH_BRIGHTER = DATA_IN / "bench_brighter_eng.parquet"

print(f'Simple source   : "{SIMPLE_CSV}"  exists={SIMPLE_CSV.exists()}')
print(f'BRIGHER source  : "{BRIGHTER_DF}"  exists={BRIGHTER_DF.exists()}')

print(f'Simple Bench   : "{BENCH_SIMPLE}"  exists={BENCH_SIMPLE.exists()}')
print(f'BRIGHER Bench  : "{BENCH_BRIGHTER}"  exists={BENCH_BRIGHTER.exists()}')


Simple source   : "/Users/stuartgow/PhD Project/Repo asa_research_prototype/data_in/simple_text_ekman6_raw.csv"  exists=True
BRIGHER source  : "/Users/stuartgow/PhD Project/Repo asa_research_prototype/data_in/brighter_emotions_raw.parquet"  exists=True
Simple Bench   : "/Users/stuartgow/PhD Project/Repo asa_research_prototype/data_in/bench_simple_ekman6.parquet"  exists=True
BRIGHER Bench  : "/Users/stuartgow/PhD Project/Repo asa_research_prototype/data_in/bench_brighter_eng.parquet"  exists=True


In [2]:
# Hugging Face Dataset - BRIGHTER
# NB uses 'joy' not 'happiness' as used by Eckman6

# import pandas as pd
from datasets import concatenate_datasets, load_dataset

# Extract all the BRIGHTER splits and add source
splits = load_dataset(HF_REPO, "eng")
raw_ds = concatenate_datasets([d.add_column("split", [name] * len(d)) for name, d in splits.items()])
# raw_ds = raw_ds.remove_columns("emotions").rename_column("joy", "happiness")
raw_ds = raw_ds.remove_columns("emotions")
raw_ds = raw_ds.add_column("source", ["brighter-eng"] * len(raw_ds))
raw_ds = raw_ds.select_columns(["source", "split", "id", "text", "anger", "disgust",
                                "fear", "joy", "sadness", "surprise"])

raw_ds.to_parquet(BRIGHTER_DF)

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

1543737

In [3]:
# Quick look at the source data
#

simple_df = pd.read_csv(SIMPLE_CSV)
brighter_df = pd.read_parquet(BRIGHTER_DF)

display(simple_df.shape)
display(simple_df.head(10))

display(brighter_df.shape)
display(brighter_df.head(10))

(20, 2)

,text,emotion
0,I am so happy,happiness
1,I'm absolutely delighted,happiness
2,I was gutted,sadness
3,I feel miserable today,sadness
4,that is absolutely revolting,disgust
5,I'm repulsed by it,disgust
6,I'm furious about it,anger
7,that made me really angry,anger
8,I was terrified,fear
9,I'm scared of what happens next,fear


(8522, 10)

,source,split,id,text,anger,disgust,fear,joy,sadness,surprise
0,brighter-eng,train,eng_train_track_a_00001,"colorado, middle of nowhere.",0,NaN,1,0,0,1
1,brighter-eng,train,eng_train_track_a_00002,this involved swimming a pretty large lake tha...,0,NaN,1,0,0,0
2,brighter-eng,train,eng_train_track_a_00003,it was one of my most shameful experiences.,0,NaN,1,0,1,0
3,brighter-eng,train,eng_train_track_a_00004,"after all, i had vegetables coming out my ears...",0,NaN,0,0,0,0
4,brighter-eng,train,eng_train_track_a_00005,then the screaming started.,0,NaN,1,0,1,1
5,brighter-eng,train,eng_train_track_a_00006,"they don't fear death, and it seems they belie...",0,NaN,1,0,0,1
6,brighter-eng,train,eng_train_track_a_00007,you know what happens when i get one of these ...,1,NaN,1,0,0,0
7,brighter-eng,train,eng_train_track_a_00008,my stomach even started giving me fits.,0,NaN,1,0,1,0
8,brighter-eng,train,eng_train_track_a_00009,"well, as we're bowling, my dinner began to not...",0,NaN,1,0,0,0
9,brighter-eng,train,eng_train_track_a_00010,hondas are notoriously great cars for long tri...,0,NaN,0,1,0,0


## Build & Save The Benchmarks



In [4]:
# Reshape a labelled corpus into the standard benchmark frame
#

from collections.abc import Mapping

from asa.core.representations import EKMAN6, AffectRepresentation

BENCH_COLUMNS = ("source", "split", "id", "text")


def to_benchmark(source_df: pd.DataFrame,
                 rep: AffectRepresentation,
                 *,
                 corpus: str,
                 split: str | None = None,
                 label: str | None = None,
                 axis_map: Mapping[str, str] | None = None) -> pd.DataFrame:
    """Reshape a labelled corpus into the standard benchmark frame.

    Columns, in order: ``source``, ``split``, ``id``, ``text``, then one float column per
    axis of *rep*, in the representation's own axis order. Nothing else is carried — a
    corpus's remaining columns belong in its raw file.

    A **single-label** corpus names its label column in *label* and is one-hot expanded,
    rows with no label becoming all-rest. A **multilabel** corpus already has a column per
    axis, and is only renamed, checked and ordered.

    **An axis the corpus never labels stays NaN, and that is deliberate.** ``rest`` is not
    "unknown": it is the positive claim that the emotion is absent. Filling it here would
    write a measurement nobody made into a file that later analysis trusts, and would hide
    a known limit of the corpus. Whoever needs an ``AffectVector`` says what they assume,
    at their own call site.

    *axis_map* renames corpus columns onto axis names, and is a **theoretical claim rather
    than a rename**: mapping BRIGHTER's ``joy`` onto ``ekman6/1``'s ``happiness`` asserts
    that the two label the same construct. It is an argument here rather than a mutation
    upstream so that the claim is visible where it is made, and it is recorded in the
    frame's ``attrs`` — a frame whose axes have been remapped is no longer strictly the
    corpus it came from.

    *split* fills that column for a corpus that does not carry one, and *corpus* both fills
    ``source`` and seeds a deterministic ``id`` where there is none — so re-running prep
    cannot renumber a row that a result already cites.
    """
    frame = source_df.copy()
    if axis_map:
        frame = frame.rename(columns=dict(axis_map), errors="raise")

    axes = [str(axis) for axis in rep.axes]
    low, high = rep.value_range

    if "text" not in frame.columns:
        raise ValueError("a benchmark frame needs a 'text' column")

    if label is not None:
        marked = frame[label].astype("string").fillna("none")
        unknown = set(marked.unique()) - set(axes) - {"none"}
        if unknown:
            raise ValueError(f"{label!r} holds values that are not axes of {rep.id}: {sorted(unknown)}")
        for axis in axes:
            # map rather than astype(float): 1.0/0.0 is only coincidentally right for
            # ekman6/1, and would silently break on a representation whose rest is not zero
            frame[axis] = (marked == axis).map({True: high, False: rep.rest})

    absent_columns = [axis for axis in axes if axis not in frame.columns]
    if absent_columns:
        raise ValueError(f"{rep.id} axes are not columns of this frame: {absent_columns} — "
                         f"pass label= if the corpus carries one label column")

    for axis in axes:
        column = frame[axis].astype(float)
        labelled = column.dropna()
        if len(labelled) and not labelled.between(low, high).all():
            raise ValueError(f"{axis!r} has values outside {rep.value_range}")
        frame[axis] = column

    if "source" not in frame.columns:
        frame["source"] = corpus
    if split is not None:
        frame["split"] = split
    if "id" not in frame.columns:
        frame["id"] = [f"{corpus}_{n:05d}" for n in range(len(frame))]

    for required in BENCH_COLUMNS:
        if required not in frame.columns:
            raise ValueError(f"no {required!r} column, and none supplied")
    if not frame["id"].is_unique:
        raise ValueError(f"{corpus} has duplicate ids")

    frame = frame[[*BENCH_COLUMNS, *axes]].reset_index(drop=True)
    frame.attrs = {"corpus": corpus,
                   "representation": rep.id,
                   "axis_map": dict(axis_map or {}),
                   "unannotated_axes": [axis for axis in axes if frame[axis].isna().all()],
                   "rows": len(frame)}
    return frame

In [5]:
# Build both benchmark frames and persist them
#

simple_bench = to_benchmark(simple_df, EKMAN6, corpus="simple-ekman6", split="all", label="emotion")
brighter_bench = to_benchmark(brighter_df, EKMAN6, corpus="brighter-eng", axis_map={"joy": "happiness"})

display(simple_bench.shape)
display(simple_bench.head(10))

display(brighter_bench.shape)
display(brighter_bench.head(10))

simple_bench.to_parquet(BENCH_SIMPLE, index=False)
brighter_bench.to_parquet(BENCH_BRIGHTER, index=False)

print(f"wrote {BENCH_SIMPLE.name}  {simple_bench.shape}")
print(f"wrote {BENCH_BRIGHTER.name}  {brighter_bench.shape}")



(20, 10)

,source,split,id,text,anger,disgust,fear,happiness,sadness,surprise
0,simple-ekman6,all,simple-ekman6_00000,I am so happy,0.0,0.0,0.0,1.0,0.0,0.0
1,simple-ekman6,all,simple-ekman6_00001,I'm absolutely delighted,0.0,0.0,0.0,1.0,0.0,0.0
2,simple-ekman6,all,simple-ekman6_00002,I was gutted,0.0,0.0,0.0,0.0,1.0,0.0
3,simple-ekman6,all,simple-ekman6_00003,I feel miserable today,0.0,0.0,0.0,0.0,1.0,0.0
4,simple-ekman6,all,simple-ekman6_00004,that is absolutely revolting,0.0,1.0,0.0,0.0,0.0,0.0
5,simple-ekman6,all,simple-ekman6_00005,I'm repulsed by it,0.0,1.0,0.0,0.0,0.0,0.0
6,simple-ekman6,all,simple-ekman6_00006,I'm furious about it,1.0,0.0,0.0,0.0,0.0,0.0
7,simple-ekman6,all,simple-ekman6_00007,that made me really angry,1.0,0.0,0.0,0.0,0.0,0.0
8,simple-ekman6,all,simple-ekman6_00008,I was terrified,0.0,0.0,1.0,0.0,0.0,0.0
9,simple-ekman6,all,simple-ekman6_00009,I'm scared of what happens next,0.0,0.0,1.0,0.0,0.0,0.0


(8522, 10)

,source,split,id,text,anger,disgust,fear,happiness,sadness,surprise
0,brighter-eng,train,eng_train_track_a_00001,"colorado, middle of nowhere.",0.0,NaN,1.0,0.0,0.0,1.0
1,brighter-eng,train,eng_train_track_a_00002,this involved swimming a pretty large lake tha...,0.0,NaN,1.0,0.0,0.0,0.0
2,brighter-eng,train,eng_train_track_a_00003,it was one of my most shameful experiences.,0.0,NaN,1.0,0.0,1.0,0.0
3,brighter-eng,train,eng_train_track_a_00004,"after all, i had vegetables coming out my ears...",0.0,NaN,0.0,0.0,0.0,0.0
4,brighter-eng,train,eng_train_track_a_00005,then the screaming started.,0.0,NaN,1.0,0.0,1.0,1.0
5,brighter-eng,train,eng_train_track_a_00006,"they don't fear death, and it seems they belie...",0.0,NaN,1.0,0.0,0.0,1.0
6,brighter-eng,train,eng_train_track_a_00007,you know what happens when i get one of these ...,1.0,NaN,1.0,0.0,0.0,0.0
7,brighter-eng,train,eng_train_track_a_00008,my stomach even started giving me fits.,0.0,NaN,1.0,0.0,1.0,0.0
8,brighter-eng,train,eng_train_track_a_00009,"well, as we're bowling, my dinner began to not...",0.0,NaN,1.0,0.0,0.0,0.0
9,brighter-eng,train,eng_train_track_a_00010,hondas are notoriously great cars for long tri...,0.0,NaN,0.0,1.0,0.0,0.0


wrote bench_simple_ekman6.parquet  (20, 10)
wrote bench_brighter_eng.parquet  (8522, 10)


# Analyse The Benchmarks

In [6]:
# What each benchmark file carries with it
#
# Read BACK from disk deliberately, so this shows what survived the write rather than what
# was still in memory. attrs travels inside the parquet as PANDAS_ATTRS key-value metadata,
# so provenance cannot be separated from the data it describes — no sidecar to lose. Caveat
# worth knowing: attrs is experimental in pandas and PANDAS_ATTRS is a pandas convention,
# so another tool reading these files sees the bytes but will not populate anything.


def show_provenance(*paths: Path) -> pd.DataFrame:
    """One row per benchmark file: where it came from, and what was done to it."""
    rows = []
    for path in paths:
        frame = pd.read_parquet(path)
        attrs = frame.attrs
        rows.append({
            "corpus": attrs.get("corpus", "— none recorded —"),
            "representation": attrs.get("representation", "—"),
            "axis_map": ", ".join(f"{was} -> {now}" for was, now in attrs.get("axis_map", {}).items()) or "—",
            "unannotated_axes": ", ".join(attrs.get("unannotated_axes", [])) or "—",
            "rows": len(frame),
            "splits": ", ".join(sorted(frame["split"].unique())),
        })
    return pd.DataFrame(rows, index=[path.name for path in paths]).rename_axis("file")


show_provenance(BENCH_SIMPLE, BENCH_BRIGHTER)

,corpus,representation,axis_map,unannotated_axes,rows,splits
file,,,,,,
bench_simple_ekman6.parquet,simple-ekman6,ekman6/1,—,—,20,all
bench_brighter_eng.parquet,brighter-eng,ekman6/1,joy -> happiness,disgust,8522,"dev, test, train"


In [7]:
# Per-axis coverage — positives, rows at rest, and rows nobody measured
#
# The 'unmeasured' column is the point of the NaN decision: at_rest is "the annotators
# looked and the emotion was not there", unmeasured is "nobody looked". Collapse them and
# a scorer counts every disgust prediction against BRIGHTER-eng as a false positive
# on a class that was never labelled.

coverage = []
for path in (BENCH_SIMPLE, BENCH_BRIGHTER):
    frame = pd.read_parquet(path)
    for axis in (str(axis) for axis in EKMAN6.axes):
        coverage.append({"file": path.name, "axis": axis,
                         "positive": int((frame[axis] > EKMAN6.rest).sum()),
                         "at_rest": int((frame[axis] == EKMAN6.rest).sum()),
                         "unmeasured": int(frame[axis].isna().sum())})

pd.DataFrame(coverage).pivot(index="axis", columns="file",
                             values=["positive", "at_rest", "unmeasured"])

positive                              \
file      bench_brighter_eng.parquet bench_simple_ekman6.parquet   
axis                                                               
anger                           1006                           2   
disgust                            0                           2   
fear                            4822                           3   
happiness                       2075                           4   
sadness                         2702                           4   
surprise                        2498                           2   

                             at_rest                              \
file      bench_brighter_eng.parquet bench_simple_ekman6.parquet   
axis                                                               
anger                           7516                          18   
disgust                            0                          18   
fear                            3700                          17   
happiness                       6447                          16   
sadness                         5820                          16   
surprise                        6024                          18   

                          unmeasured                              
file      bench_brighter_eng.parquet bench_simple_ekman6.parquet  
axis                                                              
anger                              0                           0  
disgust                         8522                           0  
fear                               0                           0  
happiness                          0                           0  
sadness                            0                           0  
surprise                           0                           0